# OmniGen2 — Multi-Image VTO Server

Hosts OmniGen2 locally on Colab GPU for multi-image virtual try-on.
Accepts person image(s) + garment image for identity-preserving try-on.

**GPU:** T4 works with CPU offload (~17GB model, T4 has 15GB). A100 is faster.

**How to use:**
1. Run all cells (~5-8 min for model download)
2. Copy the `https://xxxxx.gradio.live` URL
3. Set it as `OMNIGEN_COLAB_URL` secret in Supabase

In [ ]:
#@title 1. Install Dependencies
!pip install -q torch==2.6.0 torchvision --extra-index-url https://download.pytorch.org/whl/cu124
!pip install -q gradio Pillow requests
!pip install -q timm einops accelerate transformers==4.51.3 diffusers
!pip install -q opencv-python-headless scipy omegaconf

# Clone OmniGen2
import os
if not os.path.exists('OmniGen2'):
    !git clone https://github.com/VectorSpaceLab/OmniGen2.git
os.chdir('OmniGen2')

print('\nDependencies installed!')

In [ ]:
#@title 2. Load OmniGen2 model
import torch
import sys
sys.path.insert(0, '.')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name()} ({vram:.1f} GB VRAM)')

from pipeline import OmniGen2Pipeline

model_path = 'OmniGen2/OmniGen2'
print(f'Loading OmniGen2 from {model_path} (this takes several minutes)...')

pipe = OmniGen2Pipeline.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
)

# Enable CPU offload for T4 (15GB VRAM, model needs ~17GB)
if device == 'cuda' and vram < 20:
    print('Enabling model CPU offload for T4...')
    pipe.enable_model_cpu_offload()
else:
    pipe = pipe.to(device)

print('OmniGen2 loaded!')

In [ ]:
#@title 3. Define generation function
from PIL import Image, ImageOps
import io
import base64
import requests
import tempfile
import json

def download_image(url, max_dim=768):
    """Download image from URL, resize, return PIL Image."""
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    img = Image.open(io.BytesIO(response.content)).convert('RGB')
    img = ImageOps.exif_transpose(img)
    if max(img.size) > max_dim:
        ratio = max_dim / max(img.size)
        img = img.resize((int(img.width * ratio), int(img.height * ratio)), Image.LANCZOS)
    return img

def generate_tryon(selfie_url, fullbody_url, garment_url, description='clothing item', steps=50):
    """Run OmniGen2 virtual try-on."""
    try:
        input_images = []
        
        # Full body (required) — this is the main person image to edit
        print('Downloading full body image...')
        fullbody_img = download_image(fullbody_url)
        input_images.append(fullbody_img)
        
        # Garment (required) — reference image
        print('Downloading garment image...')
        garment_img = download_image(garment_url)
        input_images.append(garment_img)
        
        # Selfie for extra face reference (optional)
        if selfie_url and selfie_url.strip():
            print('Downloading selfie...')
            selfie_img = download_image(selfie_url)
            input_images.append(selfie_img)
        
        # Build instruction for OmniGen2
        if len(input_images) == 3:
            instruction = (
                f'Put the {description} from the second image onto the person in the first image. '
                f'Use the third image as face reference. '
                f'Preserve the person\'s exact face, skin tone, hair, and body proportions. '
                f'Natural pose, studio lighting, fashion photography.'
            )
        else:
            instruction = (
                f'Put the {description} from the second image onto the person in the first image. '
                f'Preserve the person\'s exact face, skin tone, hair, and body proportions. '
                f'Natural pose, studio lighting, fashion photography.'
            )
        
        print(f'Running OmniGen2 with {len(input_images)} images...')
        print(f'Instruction: {instruction[:100]}...')
        
        output = pipe(
            prompt=instruction,
            input_images=input_images,
            num_inference_steps=int(steps),
            text_guidance_scale=5.0,
            image_guidance_scale=1.8,
            num_images_per_prompt=1,
        )
        
        result_img = output[0] if isinstance(output, list) else output.images[0]
        
        # Convert to base64
        buf = io.BytesIO()
        result_img.save(buf, format='JPEG', quality=90)
        b64 = base64.b64encode(buf.getvalue()).decode()
        
        print('Generation complete!')
        return json.dumps({"success": True, "image_base64": b64})
    
    except Exception as e:
        import traceback
        traceback.print_exc()
        return json.dumps({"success": False, "error": str(e)})

print('Generation function ready!')

In [ ]:
#@title 4. Launch Gradio API Server
import gradio as gr

demo = gr.Interface(
    fn=generate_tryon,
    inputs=[
        gr.Textbox(label='Selfie URL (optional)', placeholder='https://...'),
        gr.Textbox(label='Full Body URL (required)', placeholder='https://...'),
        gr.Textbox(label='Garment URL (required)', placeholder='https://...'),
        gr.Textbox(label='Description', value='clothing item'),
        gr.Slider(minimum=20, maximum=80, value=50, step=5, label='Steps'),
    ],
    outputs=gr.Textbox(label='Result JSON'),
    title='OmniGen2 VTO Server',
    description='Virtual Try-On powered by OmniGen2. Send person + garment image URLs.',
    api_name='tryon',
)

print('\n' + '='*60)
print('STARTING OMNIGEN2 SERVER...')
print('Copy the public URL below and set it as')
print('OMNIGEN_COLAB_URL in your Supabase secrets.')
print('='*60 + '\n')

demo.launch(share=True, quiet=False)